In [19]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
import datetime

DARPA_DIR  = Path("../data/darpa")
OUTPUT_DIR = Path("../data/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CDM = "com.bbn.tc.schema.avro.cdm18"
SAMPLE_SIZE = 50_000 

json_files = sorted([f for f in DARPA_DIR.rglob("*.json") if f.is_file()])
print(f"Files: {len(json_files)}")
for f in json_files:
    print(f"  {f.name} ({f.stat().st_size / 1e6:.0f} MB)")

Files: 3
  ta1-cadets-e3-official-1.json (4303 MB)
  ta1-cadets-e3-official-2.json (4299 MB)
  ta1-cadets-e3-official.json (4306 MB)


In [20]:
# Pass 1: build maps of entities (subjects, files, netflows) for quick lookup
subject_map  = {}
file_map     = {}
netflow_map  = {}

print(f"Pass 1: building entity maps from {json_files[0].name}...")
print("(This will take several minutes for a 4GB file)")

with open(json_files[0], "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i % 500_000 == 0:
            print(f"  {i:,} lines | subjects={len(subject_map):,} files={len(file_map):,} netflows={len(netflow_map):,}")
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        datum = rec.get("datum", {})
        if not isinstance(datum, dict):
            continue

        s = datum.get(f"{CDM}.Subject")
        if s:
            uid = s.get("uuid")
            props = unwrap(s.get("properties")) or {}
            subject_map[uid] = {
                "cmdLine": unwrap(s.get("cmdLine")) or props.get("exec", "unknown"),
                "pid":     unwrap(s.get("pid")),
                "ppid":    unwrap(s.get("ppid")),
            }

        fo = datum.get(f"{CDM}.FileObject")
        if fo:
            uid = fo.get("uuid")
            props = unwrap(fo.get("properties")) or {}
            file_map[uid] = props.get("path", unwrap(fo.get("url")) or "unknown_file")

        nf = datum.get(f"{CDM}.NetFlowObject")
        if nf:
            uid = nf.get("uuid")
            netflow_map[uid] = {
                "local":  f"{nf.get('localAddress','?')}:{nf.get('localPort','?')}",
                "remote": f"{nf.get('remoteAddress','?')}:{nf.get('remotePort','?')}",
            }

print(f"\nPass 1 complete:")
print(f"  Subjects:   {len(subject_map):,}")
print(f"  Files:      {len(file_map):,}")
print(f"  NetFlows:   {len(netflow_map):,}")

# Pass 2: load events from the first N lines only (manageable sample)
raw_events = []
SAMPLE_SIZE = 200_000

print(f"\nPass 2: loading {SAMPLE_SIZE:,} events for analysis...")
with open(json_files[0], "r", encoding="utf-8") as f:
    lines_read = 0
    for line in f:
        if lines_read >= SAMPLE_SIZE:
            break
        line = line.strip()
        if not line:
            continue
        lines_read += 1
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        datum = rec.get("datum", {})
        ev = datum.get(f"{CDM}.Event") if isinstance(datum, dict) else None
        if ev:
            raw_events.append(ev)

print(f"Events loaded: {len(raw_events):,}")

Pass 1: building entity maps from ta1-cadets-e3-official-1.json...
(This will take several minutes for a 4GB file)
  0 lines | subjects=0 files=0 netflows=0
  500,000 lines | subjects=2,484 files=20,757 netflows=3,836
  1,000,000 lines | subjects=4,331 files=35,675 netflows=13,939
  1,500,000 lines | subjects=6,857 files=58,758 netflows=16,556
  2,000,000 lines | subjects=9,179 files=87,943 netflows=20,715
  2,500,000 lines | subjects=10,017 files=139,322 netflows=20,955
  3,000,000 lines | subjects=12,352 files=171,507 netflows=21,597
  3,500,000 lines | subjects=15,299 files=199,949 netflows=22,428
  4,000,000 lines | subjects=18,218 files=227,881 netflows=23,250
  4,500,000 lines | subjects=20,376 files=246,539 netflows=29,088

Pass 1 complete:
  Subjects:   22,306
  Files:      263,093
  NetFlows:   37,683

Pass 2: loading 200,000 events for analysis...
Events loaded: 187,951


In [21]:
# Resolve events

def resolve_uuid(uid):
    if uid in subject_map:
        cmd = subject_map[uid]["cmdLine"] or "unknown_process"
        return f"[proc] {cmd[:50]}"
    if uid in file_map:
        return f"[file] {file_map[uid][:60]}"
    if uid in netflow_map:
        nf = netflow_map[uid]
        return f"[net]  {nf['local']} -> {nf['remote']}"
    return f"[?]    {uid[:16]}..."

def ns_to_ts(ns):
    if ns is None:
        return None
    try:
        return datetime.datetime.utcfromtimestamp(int(ns) / 1e9).strftime("%Y-%m-%d %H:%M:%S")
    except:
        return None

resolved = []
for ev in raw_events:
    subj_uid = (ev.get("subject") or {}).get(f"{CDM}.UUID")
    obj_uid  = (ev.get("predicateObject") or {}).get(f"{CDM}.UUID")
    etype    = ev.get("type", "UNKNOWN")
    ts       = ns_to_ts(ev.get("timestampNanos"))
    name     = unwrap(ev.get("name")) or ""

    resolved.append({
        "timestamp":    ts,
        "event_type":   etype,
        "syscall":      name,
        "subject":      resolve_uuid(subj_uid) if subj_uid else None,
        "object":       resolve_uuid(obj_uid)  if obj_uid  else None,
        "subject_uuid": subj_uid,
        "object_uuid":  obj_uid,
    })

df = pd.DataFrame(resolved)
print(f"Resolved events: {len(df):,}")
print(f"\nEvent type distribution:")
print(df["event_type"].value_counts().to_string())

Resolved events: 187,951

Event type distribution:
event_type
EVENT_READ                      56501
EVENT_CLOSE                     23310
EVENT_MMAP                      22550
EVENT_LSEEK                     18812
EVENT_FCNTL                     18688
EVENT_WRITE                     14918
EVENT_OPEN                      14254
EVENT_CREATE_OBJECT              3473
EVENT_CHANGE_PRINCIPAL           2636
EVENT_SENDTO                     2453
EVENT_RECVFROM                   2336
EVENT_CONNECT                    1262
EVENT_MODIFY_PROCESS             1123
EVENT_FORK                        941
EVENT_EXIT                        872
EVENT_EXECUTE                     832
EVENT_ADD_OBJECT_ATTRIBUTE        791
EVENT_ACCEPT                      642
EVENT_UNLINK                      494
EVENT_MODIFY_FILE_ATTRIBUTES      472
EVENT_LINK                        162
EVENT_MPROTECT                    102
EVENT_TRUNCATE                     97
EVENT_RENAME                       74
EVENT_LOGIN               

In [22]:
# Focusing on network events
NETWORK_EVENTS = {"EVENT_CONNECT", "EVENT_SENDTO", "EVENT_RECVFROM",
                  "EVENT_ACCEPT", "EVENT_RECVMSG", "EVENT_SENDMSG"}

net_df = df[df["event_type"].isin(NETWORK_EVENTS)].copy()
print(f"Network events: {len(net_df)}")

if not net_df.empty:
    print()
    for _, row in net_df.head(20).iterrows():
        print(f"  [{row['timestamp']}] {row['event_type']}")
        print(f"    subject: {row['subject']}")
        print(f"    object:  {row['object']}")
        print()
else:
    print("No network events in this sample window — they are sparse.")
    print("Try scanning a larger sample or a different file section.")

Network events: 6770

  [2018-04-06 18:01:22] EVENT_CONNECT
    subject: [proc] unknown
    object:  [file] unknown_file

  [2018-04-06 18:01:22] EVENT_SENDTO
    subject: [proc] unknown
    object:  [file] unknown_file

  [2018-04-06 18:01:22] EVENT_RECVFROM
    subject: [proc] unknown
    object:  [file] unknown_file

  [2018-04-06 18:01:25] EVENT_CONNECT
    subject: [proc] unknown
    object:  [file] unknown_file

  [2018-04-06 18:01:25] EVENT_SENDTO
    subject: [proc] unknown
    object:  [file] unknown_file

  [2018-04-06 18:01:25] EVENT_RECVFROM
    subject: [proc] unknown
    object:  [file] unknown_file

  [2018-04-06 18:01:25] EVENT_CONNECT
    subject: [proc] unknown
    object:  [file] unknown_file

  [2018-04-06 18:01:25] EVENT_SENDTO
    subject: [proc] unknown
    object:  [file] unknown_file

  [2018-04-06 18:01:25] EVENT_RECVFROM
    subject: [proc] unknown
    object:  [file] unknown_file

  [2018-04-06 18:01:32] EVENT_CONNECT
    subject: [proc] unknown
    object: 

In [23]:
# Most active processes 
proc_counts = df[df["subject"].str.startswith("[proc]", na=False)]["subject"].value_counts()
print("Most active processes (by event count):")
print(proc_counts.head(20).to_string())

Most active processes (by event count):
subject
[proc] unknown    187160


In [24]:
# Execution chains: fork/exec/exit events that link processes together
exec_df = df[df["event_type"].isin({"EVENT_FORK", "EVENT_EXECUTE", "EVENT_EXIT"})].copy()
print(f"Execution chain events: {len(exec_df)}")
print()
for _, row in exec_df.head(30).iterrows():
    print(f"  [{row['timestamp']}] {row['event_type']:20s} | {row['subject']} -> {row['object']}")

Execution chain events: 2645

  [2018-04-06 18:01:22] EVENT_FORK           | [proc] unknown -> [proc] unknown
  [2018-04-06 18:01:22] EVENT_EXECUTE        | [proc] unknown -> [file] unknown_file
  [2018-04-06 18:01:22] EVENT_EXECUTE        | [proc] unknown -> [file] unknown_file
  [2018-04-06 18:01:22] EVENT_EXECUTE        | [proc] unknown -> [file] unknown_file
  [2018-04-06 18:01:22] EVENT_FORK           | [proc] unknown -> [proc] unknown
  [2018-04-06 18:01:22] EVENT_FORK           | [proc] unknown -> [proc] unknown
  [2018-04-06 18:01:22] EVENT_FORK           | [proc] unknown -> [proc] unknown
  [2018-04-06 18:01:22] EVENT_FORK           | [proc] unknown -> [proc] unknown
  [2018-04-06 18:01:22] EVENT_EXECUTE        | [proc] unknown -> [file] unknown_file
  [2018-04-06 18:01:22] EVENT_EXECUTE        | [proc] unknown -> [file] unknown_file
  [2018-04-06 18:01:22] EVENT_EXECUTE        | [proc] unknown -> [file] unknown_file
  [2018-04-06 18:01:22] EVENT_EXIT           | [proc] unknow

In [25]:
# file access patterns
file_events = df[
    df["event_type"].isin({"EVENT_READ", "EVENT_WRITE", "EVENT_OPEN", "EVENT_UNLINK"}) &
    df["object"].str.startswith("[file]", na=False)
].copy()

print(f"File access events: {len(file_events):,}")
print()
print("Most accessed files:")
print(file_events["object"].value_counts().head(20).to_string())

print()
SENSITIVE = ["/etc/passwd", "/etc/shadow", "/.ssh", "/etc/sudoers",
             "/root", "/tmp", "/var/log", "id_rsa"]
sensitive_hits = file_events[
    file_events["object"].apply(lambda x: any(s in str(x) for s in SENSITIVE))
]
print(f"Sensitive file accesses: {len(sensitive_hits)}")
if not sensitive_hits.empty:
    print(sensitive_hits[["timestamp", "event_type", "subject", "object"]].to_string(index=False))

File access events: 81,799

Most accessed files:
object
[file] unknown_file    81799

Sensitive file accesses: 0


In [26]:
# Time range of this sample

df_timed = df[df["timestamp"].notna()]
if not df_timed.empty:
    print(f"Sample time range:")
    print(f"  Start: {df_timed['timestamp'].min()}")
    print(f"  End:   {df_timed['timestamp'].max()}")
    print(f"  Total events with timestamps: {len(df_timed):,}")

# Events per minute — shows activity spikes
df_timed = df_timed.copy()
df_timed["minute"] = df_timed["timestamp"].str[:16]
per_minute = df_timed.groupby("minute").size().reset_index(name="count")
print(f"\nEvents per minute (top 10 busiest):")
print(per_minute.sort_values("count", ascending=False).head(10).to_string(index=False))

Sample time range:


  Start: 2018-04-06 18:01:12
  End:   2018-04-06 18:52:27
  Total events with timestamps: 187,951

Events per minute (top 10 busiest):
          minute  count
2018-04-06 18:43  15138
2018-04-06 18:42   9506
2018-04-06 18:41   8329
2018-04-06 18:15   8187
2018-04-06 18:29   7588
2018-04-06 18:14   6971
2018-04-06 18:05   6304
2018-04-06 18:02   5382
2018-04-06 18:12   5191
2018-04-06 18:44   5124
